# 🧠 Smart AI Calculator: Handwritten Math Symbol Recognition
### Complete Training Pipeline & TensorFlow Lite Export (22 Classes)

This notebook trains an ultra-lightweight Convolutional Neural Network (CNN) to recognize 22 mathematical symbols (digits 0-9, operators, variables x, y, and exponents) for on-device inference on Android.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

In [ ]:
# Step 1: Install & Import Dependencies
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {bool(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# Step 2: Define 22 Mathematical Classes
CLASSES = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    '+', '-', '*', '/', '=', '.', '(', ')', 'x', 'y', '^', ','
]
NUM_CLASSES = len(CLASSES)
LABEL_TO_INDEX = {lbl: i for i, lbl in enumerate(CLASSES)}
INDEX_TO_LABEL = {i: lbl for i, lbl in enumerate(CLASSES)}

print(f"Total classes: {NUM_CLASSES}")
print("Classes:", CLASSES)

In [ ]:
# Step 3: Dataset Generation & Augmentation
def generate_synthetic_symbol(symbol: str, size: int = 28) -> np.ndarray:
    img = np.zeros((size, size), dtype=np.float32)
    center = size / 2.0
    thickness = random.randint(2, 3)
    cx = center + random.uniform(-2.0, 2.0)
    cy = center + random.uniform(-2.0, 2.0)

    def draw_line(x1, y1, x2, y2, stroke_w=thickness):
        dist = math.hypot(x2 - x1, y2 - y1)
        steps = max(int(dist * 2), 1)
        for s in range(steps + 1):
            t = s / steps
            px = int(round(x1 + t * (x2 - x1)))
            py = int(round(y1 + t * (y2 - y1)))
            for dy in range(-stroke_w, stroke_w + 1):
                for dx in range(-stroke_w, stroke_w + 1):
                    if dx * dx + dy * dy <= stroke_w * stroke_w:
                        nx, ny = px + dx, py + dy
                        if 0 <= nx < size and 0 <= ny < size:
                            img[ny, nx] = min(255.0, img[ny, nx] + random.uniform(200.0, 255.0))

    if symbol == '+':
        draw_line(cx - 7, cy, cx + 7, cy)
        draw_line(cx, cy - 7, cx, cy + 7)
    elif symbol == '-':
        draw_line(cx - 8, cy, cx + 8, cy)
    elif symbol == '*':
        draw_line(cx - 6, cy - 6, cx + 6, cy + 6)
        draw_line(cx - 6, cy + 6, cx + 6, cy - 6)
    elif symbol == '/':
        draw_line(cx - 6, cy + 8, cx + 6, cy - 8)
    elif symbol == '=':
        draw_line(cx - 8, cy - 3, cx + 8, cy - 3)
        draw_line(cx - 8, cy + 3, cx + 8, cy + 3)
    elif symbol == '.':
        for dy in range(-2, 3):
            for dx in range(-2, 3):
                if dx * dx + dy * dy <= 4:
                    ny, nx = int(cy + 6 + dy), int(cx + dx)
                    if 0 <= nx < size and 0 <= ny < size:
                        img[ny, nx] = 255.0
    elif symbol == '(':
        for deg in range(120, 241, 10):
            rad = math.radians(deg)
            draw_line(cx + 2 + 7 * math.cos(rad), cy + 9 * math.sin(rad), cx + 2 + 7 * math.cos(rad), cy + 9 * math.sin(rad), stroke_w=2)
    elif symbol == ')':
        for deg in range(-60, 61, 10):
            rad = math.radians(deg)
            draw_line(cx - 2 + 7 * math.cos(rad), cy + 9 * math.sin(rad), cx - 2 + 7 * math.cos(rad), cy + 9 * math.sin(rad), stroke_w=2)
    elif symbol == 'x':
        draw_line(cx - 6, cy - 6, cx + 6, cy + 6)
        draw_line(cx - 6, cy + 6, cx + 6, cy - 6)
    elif symbol == 'y':
        draw_line(cx - 6, cy - 7, cx, cy + 1)
        draw_line(cx + 6, cy - 7, cx - 4, cy + 8)
    elif symbol == '^':
        draw_line(cx - 6, cy + 3, cx, cy - 6)
        draw_line(cx, cy - 6, cx + 6, cy + 3)
    elif symbol == ',':
        draw_line(cx, cy + 4, cx - 2, cy + 8)
    else:
        draw_line(cx, cy - 8, cx, cy + 8)

    noise = np.random.normal(0, 3, (size, size))
    return np.clip(img + noise, 0, 255).astype(np.float32)

# Load MNIST + Generate non-digits
(x_train_m, y_train_m), (x_test_m, y_test_m) = tf.keras.datasets.mnist.load_data()
x_all_m = np.concatenate([x_train_m, x_test_m], axis=0)
y_all_m = np.concatenate([y_train_m, y_test_m], axis=0)

images, labels = [], []
samples_per_class = 2000

# Digits
for digit in range(10):
    idx = np.where(y_all_m == digit)[0][:samples_per_class]
    images.extend(x_all_m[idx])
    labels.extend([str(digit)] * len(idx))

# Operators & Variables
for s in [c for c in CLASSES if not c.isdigit()]:
    for _ in range(samples_per_class):
        images.append(generate_synthetic_symbol(s))
        labels.append(s)

X = np.expand_dims(np.array(images, dtype=np.float32) / 255.0, axis=-1)
y = np.array([LABEL_TO_INDEX[l] for l in labels], dtype=np.int32)

perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]
print(f"Final dataset: {X.shape}, labels: {y.shape}")

In [ ]:
# Step 4: Build Lightweight CNN Architecture
model = models.Sequential([
    layers.Input(shape=(28, 28, 1), name="input_image"),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.2),

    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation='softmax', name="output_probabilities")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Step 5: Train Model
split_idx = int(0.85 * len(X))
X_train, y_train = X[:split_idx], y[:split_idx]
X_val, y_val = X[split_idx:], y[split_idx:]

cb = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=cb
)

In [ ]:
# Step 6: Convert to INT8 Quantized TensorFlow Lite Model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()
tflite_path = "math_symbol_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"Saved quantized model to '{tflite_path}' (Size: {size_mb:.2f} MB)")

# Save labels.txt
with open("labels.txt", "w") as f:
    for l in CLASSES:
        f.write(f"{l}\n")
print("Saved labels.txt")